In [0]:
from pyspark.sql.functions import col

dbutils.widgets.text("batch_id", "full-sample-20261010-01", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()

if not batch_id:
    raise ValueError("batch_id is required")

bronze_batch_df = (
    spark.table("workspace.tmdb_lakehouse.bronze_movie_records")
    .filter(col("batch_id") == batch_id)
)

batch_count = bronze_batch_df.count()
if batch_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

print(f"Bronze rows in selected batch: {batch_count}")
bronze_batch_df.select(
    "batch_id",
    "load_type",
    "source_extracted_at",
    "window_start",
    "window_end",
).distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import col, from_json, size
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

movie_schema = StructType([
    StructField("id", LongType(), True),
    StructField("imdb_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("original_title", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("original_language", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("adult", BooleanType(), True),
    StructField("video", BooleanType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("vote_average", DoubleType(), True),
    StructField("vote_count", LongType(), True),
    StructField("budget", LongType(), True),
    StructField("revenue", LongType(), True),

    StructField("genres", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ])), True),

    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("origin_country", StringType(), True),
    ])), True),

    StructField("credits", StructType([
        StructField("cast", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("credit_id", StringType(), True),
            StructField("name", StringType(), True),
            StructField("character", StringType(), True),
            StructField("order", IntegerType(), True),
        ])), True),
    ]), True),

    StructField("keywords", StructType([
        StructField("keywords", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("name", StringType(), True),
        ])), True),
    ]), True),

    StructField("reviews", StructType([
        StructField("results", ArrayType(StructType([
            StructField("id", StringType(), True),
            StructField("author", StringType(), True),
            StructField("author_details", StructType([
                StructField("name", StringType(), True),
                StructField("username", StringType(), True),
                StructField("avatar_path", StringType(), True),
                StructField("rating", DoubleType(), True),
            ]), True),
            StructField("content", StringType(), True),
            StructField("created_at", StringType(), True),
            StructField("updated_at", StringType(), True),
        ])), True),
    ]), True),
])

parsed_movies_df = (
    bronze_batch_df
    .select(
        col("movie_id").alias("bronze_movie_id"),
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        from_json(col("raw_payload"), movie_schema).alias("movie"),
    )
    .select(
        "bronze_movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        "movie.*",
    )
)

parsed_movies_df.printSchema()

print("Bronze rows in batch:", bronze_batch_df.count())
print("Parsed movie IDs missing:", parsed_movies_df.filter(col("id").isNull()).count())
print("Parsed titles missing:", parsed_movies_df.filter(col("title").isNull()).count())

display(
    parsed_movies_df.select(
        "bronze_movie_id",
        "id",
        "title",
        "budget",
        "revenue",
        size("genres").alias("genre_count"),
        size("production_companies").alias("studio_count"),
        size("credits.cast").alias("cast_count"),
        size("keywords.keywords").alias("keyword_count"),
        size("reviews.results").alias("review_count"),
    ).limit(10)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

# Stop rather than silently accepting records that fail the movie contract.
quarantine_table = "workspace.tmdb_lakehouse.pipeline_quarantine"

validated_movies_df = parsed_movies_df.withColumn(
    "validation_reason",
    F.when(
        F.col("id").isNull(),
        F.lit("Parsed movie ID is missing"),
    )
    .when(
        F.col("bronze_movie_id").isNull()
        | (F.col("id") != F.col("bronze_movie_id")),
        F.lit("Parsed movie ID does not match Bronze movie ID"),
    )
    .when(
        F.col("title").isNull()
        | (F.length(F.trim(F.col("title"))) == 0),
        F.lit("Movie title is missing or blank"),
    ),
)

invalid_movies_df = validated_movies_df.filter(
    F.col("validation_reason").isNotNull()
)

quarantine_source_df = (
    invalid_movies_df
    .withColumn(
        "record_key",
        F.coalesce(
            F.col("bronze_movie_id").cast("string"),
            F.col("id").cast("string"),
            F.lit("unknown"),
        ),
    )
    .withColumn(
        "record_hash",
        F.sha2(
            F.to_json(F.struct("bronze_movie_id", "id", "title")),
            256,
        ),
    )
    .withColumn(
        "quarantine_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.lit("Bronze-to-Silver"),
                F.coalesce(F.col("batch_id"), F.lit("")),
                F.coalesce(F.col("source_file"), F.lit("")),
                F.col("record_key"),
                F.col("record_hash"),
            ),
            256,
        ),
    )
    .select(
        "quarantine_id",
        F.lit("Bronze-to-Silver").alias("layer"),
        F.col("batch_id"),
        F.col("source_file"),
        F.col("source_extracted_at"),
        "record_key",
        F.col("validation_reason").alias("reason"),
        "record_hash",
        F.current_timestamp().alias("quarantined_at"),
    )
)

quarantine_source_df.createOrReplaceTempView("movie_quarantine_source")

spark.sql(f"""
    MERGE INTO {quarantine_table} AS target
    USING movie_quarantine_source AS source
    ON target.quarantine_id = source.quarantine_id

    WHEN NOT MATCHED THEN INSERT (
        quarantine_id,
        layer,
        batch_id,
        source_file,
        source_extracted_at,
        record_key,
        reason,
        record_hash,
        quarantined_at
    ) VALUES (
        source.quarantine_id,
        source.layer,
        source.batch_id,
        source.source_file,
        source.source_extracted_at,
        source.record_key,
        source.reason,
        source.record_hash,
        source.quarantined_at
    )
""")

valid_parsed_movies_df = (
    validated_movies_df
    .filter(F.col("validation_reason").isNull())
    .drop("validation_reason")
)

invalid_count = invalid_movies_df.count()

print("Invalid movie records quarantined:", invalid_count)
print("Valid movie rows continuing:", valid_parsed_movies_df.count())

clean_budget = F.when(F.col("budget") > 0, F.col("budget")).otherwise(
    F.lit(None).cast(LongType())
)
clean_revenue = F.when(F.col("revenue") > 0, F.col("revenue")).otherwise(
    F.lit(None).cast(LongType())
)

silver_movies_source_df = (
    valid_parsed_movies_df
    .select(
        F.col("id").alias("movie_id"),
        "imdb_id",
        "title",
        "original_title",
        "overview",
        "original_language",
        F.expr( 
            "try_to_date(NULLIF(TRIM(release_date), ''), 'yyyy-MM-dd')"
        ).alias("release_date"),
        "runtime",
        "status",
        "adult",
        "video",
        "popularity",
        "vote_average",
        "vote_count",
        clean_budget.alias("budget"),
        clean_revenue.alias("revenue"),
        F.when(
            (clean_budget.isNotNull()) & (clean_revenue.isNotNull()),
            clean_revenue - clean_budget,
        ).cast(LongType()).alias("profit"),
        F.when(
            (clean_budget.isNotNull()) & (clean_revenue.isNotNull()),
            (clean_revenue - clean_budget) / clean_budget,
        ).cast("double").alias("roi"),
        F.when(clean_budget.isNull(), F.lit(None).cast("string"))
        .when(clean_budget < 10000000, F.lit("low"))
        .when(clean_budget < 100000000, F.lit("medium"))
        .otherwise(F.lit("high"))
        .alias("budget_tier"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

business_columns = [
    "movie_id",
    "imdb_id",
    "title",
    "original_title",
    "overview",
    "original_language",
    "release_date",
    "runtime",
    "status",
    "adult",
    "video",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue",
    "profit",
    "roi",
    "budget_tier",
]

silver_movies_source_df = (
    silver_movies_source_df
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(F.struct(*[F.col(name) for name in business_columns])),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

print("Silver movie rows prepared:", silver_movies_source_df.count())
silver_movies_source_df.printSchema()

display(
    silver_movies_source_df.select(
        "movie_id",
        "title",
        "release_date",
        "budget",
        "revenue",
        "profit",
        "roi",
        "budget_tier",
        "source_batch_id",
        "source_extracted_at",
    ).limit(10)
)

In [0]:
print(
    "Unknown-budget rows with a tier:",
    silver_movies_source_df.filter(
        F.col("budget").isNull() & F.col("budget_tier").isNotNull()
    ).count()
)

In [0]:
from pyspark.sql import functions as F

duplicate_movie_keys = (
    silver_movies_source_df
    .groupBy("movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_financials = silver_movies_source_df.filter(
    (
        F.col("budget").isNotNull()
        & F.col("revenue").isNotNull()
        & (
            (F.col("profit") != F.col("revenue") - F.col("budget"))
            | (
                F.abs(
                    F.col("roi")
                    - ((F.col("revenue") - F.col("budget")) / F.col("budget"))
                ) > 1e-9
            )
        )
    )
    | (
        (F.col("budget").isNull() | F.col("revenue").isNull())
        & (F.col("profit").isNotNull() | F.col("roi").isNotNull())
    )
).count()

invalid_budget_tiers = silver_movies_source_df.filter(
    (F.col("budget").isNull() & F.col("budget_tier").isNotNull())
    | (
        (F.col("budget") > 0)
        & (F.col("budget") < 10000000)
        & (F.col("budget_tier") != "low")
    )
    | (
        (F.col("budget") >= 10000000)
        & (F.col("budget") < 100000000)
        & (F.col("budget_tier") != "medium")
    )
    | (
        (F.col("budget") >= 100000000)
        & (F.col("budget_tier") != "high")
    )
).count()

missing_hashes = silver_movies_source_df.filter(
    F.col("row_hash").isNull()
).count()

print("Rows prepared:", silver_movies_source_df.count())
print("Duplicate movie keys:", duplicate_movie_keys)
print("Invalid financial calculations:", invalid_financials)
print("Invalid budget tiers:", invalid_budget_tiers)
print("Missing row hashes:", missing_hashes)

if any([duplicate_movie_keys, invalid_financials, invalid_budget_tiers, missing_hashes]):
    raise ValueError("Silver movie validation failed; do not write the table.")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movies_cleansed (
    movie_id BIGINT,
    imdb_id STRING,
    title STRING,
    original_title STRING,
    overview STRING,
    original_language STRING,
    release_date DATE,
    runtime INT,
    status STRING,
    adult BOOLEAN,
    video BOOLEAN,
    popularity DOUBLE,
    vote_average DOUBLE,
    vote_count BIGINT,
    budget BIGINT,
    revenue BIGINT,
    profit BIGINT,
    roi DOUBLE,
    budget_tier STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movies_cleansed").show(truncate=False)

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

movies_table = "workspace.tmdb_lakehouse.silver_movies_cleansed"
logs_table = "workspace.tmdb_lakehouse.pipeline_execution_logs"

run_id = str(uuid4())
started_at = datetime.now(timezone.utc).replace(tzinfo=None)
rows_inserted = 0
rows_updated = 0
rows_quarantined = invalid_count

log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

def write_movie_run_log(status, error_details=None):
    ended_at = datetime.now(timezone.utc).replace(tzinfo=None)
    load_type = bronze_batch_df.select("load_type").first()["load_type"]

    log_row = [(
        run_id,
        "Bronze-to-Silver",
        load_type,
        f"batch_id={batch_id}; table=silver_movies_cleansed",
        batch_id,
        started_at,
        ended_at,
        status,
        rows_inserted,
        rows_updated,
        rows_quarantined,
        error_details,
        ended_at,
    )]

    (
        spark.createDataFrame(log_row, schema=log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(logs_table)
    )

try:
    source_count = silver_movies_source_df.count()
    if source_count == 0:
        raise ValueError(f"No Silver movie rows prepared for batch_id={batch_id}")

    duplicate_keys = (
        silver_movies_source_df
        .groupBy("movie_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    if duplicate_keys:
        raise ValueError(f"Found {duplicate_keys} duplicate movie_id keys")

    missing_lineage = silver_movies_source_df.filter(
        F.col("movie_id").isNull()
        | F.col("source_batch_id").isNull()
        | F.col("source_extracted_at").isNull()
    ).count()
    if missing_lineage:
        raise ValueError(f"Found {missing_lineage} rows missing required lineage")

    existing_movies_df = spark.table(movies_table)

    stored_movie_lookup_df = existing_movies_df.select(
        "movie_id",
        F.col("source_extracted_at").alias("stored_extracted_at"),
        F.col("source_batch_id").alias("stored_batch_id"),
        F.col("row_hash").alias("stored_row_hash"),
        F.col("source_file").alias("stored_source_file"),
    )

    joined_df = (
        silver_movies_source_df.alias("incoming")
        .join(
            stored_movie_lookup_df.alias("stored"),
            F.col("incoming.movie_id") == F.col("stored.movie_id"),
            "left",
        )
    )

    ambiguous_snapshots = joined_df.filter(
        F.col("stored.stored_batch_id").isNotNull()
        & (
            F.col("incoming.source_extracted_at")
            == F.col("stored.stored_extracted_at")
        )
        & (
            F.col("incoming.source_batch_id")
            != F.col("stored.stored_batch_id")
        )
    ).count()

    if ambiguous_snapshots:
        raise ValueError(
            f"{ambiguous_snapshots} movie snapshots have the same extraction "
            "timestamp but different batch IDs; resolve snapshot ordering before merging"
        )

    stale_snapshots = joined_df.filter(
        F.col("stored.stored_batch_id").isNotNull()
        & (
            F.col("incoming.source_extracted_at")
            < F.col("stored.stored_extracted_at")
        )
    ).count()

    if stale_snapshots:
        raise ValueError(
            f"{stale_snapshots} movie snapshots are older than the stored data; "
            "the merge was stopped to protect newer Silver rows"
        )

    rows_inserted = (
        silver_movies_source_df
        .join(
            existing_movies_df.select("movie_id"),
            on="movie_id",
            how="left_anti",
        )
        .count()
    )

    eligible_existing_rows = joined_df.filter(
        F.col("stored.stored_batch_id").isNotNull()
        & (
            (
                F.col("incoming.source_extracted_at")
                > F.col("stored.stored_extracted_at")
            )
            | (
                (
                    F.col("incoming.source_extracted_at")
                    == F.col("stored.stored_extracted_at")
                )
                & (
                    F.col("incoming.source_batch_id")
                    == F.col("stored.stored_batch_id")
                )
            )
        )
    )

    rows_updated = eligible_existing_rows.filter(
        ~F.col("incoming.row_hash").eqNullSafe(
            F.col("stored.stored_row_hash")
        )
        | ~F.col("incoming.source_file").eqNullSafe(
            F.col("stored.stored_source_file")
        )
        | ~F.col("incoming.source_batch_id").eqNullSafe(
            F.col("stored.stored_batch_id")
        )
    ).count()

    silver_movies_source_df.createOrReplaceTempView(
        "silver_movies_merge_source"
    )

    spark.sql(f"""
        MERGE INTO {movies_table} AS target
        USING silver_movies_merge_source AS source
        ON target.movie_id = source.movie_id

        WHEN MATCHED AND (
            source.source_extracted_at > target.source_extracted_at
            OR (
                source.source_extracted_at = target.source_extracted_at
                AND source.source_batch_id = target.source_batch_id
            )
        ) AND (
            NOT (target.row_hash <=> source.row_hash)
            OR NOT (target.source_batch_id <=> source.source_batch_id)
            OR NOT (target.source_file <=> source.source_file)
        )
        THEN UPDATE SET
            target.imdb_id = source.imdb_id,
            target.title = source.title,
            target.original_title = source.original_title,
            target.overview = source.overview,
            target.original_language = source.original_language,
            target.release_date = source.release_date,
            target.runtime = source.runtime,
            target.status = source.status,
            target.adult = source.adult,
            target.video = source.video,
            target.popularity = source.popularity,
            target.vote_average = source.vote_average,
            target.vote_count = source.vote_count,
            target.budget = source.budget,
            target.revenue = source.revenue,
            target.profit = source.profit,
            target.roi = source.roi,
            target.budget_tier = source.budget_tier,
            target.source_batch_id = source.source_batch_id,
            target.source_file = source.source_file,
            target.source_extracted_at = source.source_extracted_at,
            target.row_hash = source.row_hash,
            target.load_timestamp = CASE
                WHEN NOT (target.row_hash <=> source.row_hash)
                THEN source.load_timestamp
                ELSE target.load_timestamp
            END

        WHEN NOT MATCHED THEN INSERT (
            movie_id,
            imdb_id,
            title,
            original_title,
            overview,
            original_language,
            release_date,
            runtime,
            status,
            adult,
            video,
            popularity,
            vote_average,
            vote_count,
            budget,
            revenue,
            profit,
            roi,
            budget_tier,
            source_batch_id,
            source_file,
            source_extracted_at,
            row_hash,
            load_timestamp
        ) VALUES (
            source.movie_id,
            source.imdb_id,
            source.title,
            source.original_title,
            source.overview,
            source.original_language,
            source.release_date,
            source.runtime,
            source.status,
            source.adult,
            source.video,
            source.popularity,
            source.vote_average,
            source.vote_count,
            source.budget,
            source.revenue,
            source.profit,
            source.roi,
            source.budget_tier,
            source.source_batch_id,
            source.source_file,
            source.source_extracted_at,
            source.row_hash,
            source.load_timestamp
        )
    """)

except Exception as exc:
    try:
        write_movie_run_log("FAILURE", str(exc)[:4000])
    except Exception as log_exc:
        raise RuntimeError(
            f"Movie merge failed: {exc}; failure logging also failed: {log_exc}"
        ) from log_exc
    raise
else:
    write_movie_run_log("SUCCESS")

    print("Movie rows inserted:", rows_inserted)
    print("Movie rows updated:", rows_updated)
    print("Movie rows quarantined:", rows_quarantined)
    print("Silver movie rows now stored:", spark.table(movies_table).count())

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.pipeline_quarantine (
    quarantine_id STRING,
    layer STRING,
    batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    record_key STRING,
    reason STRING,
    record_hash STRING,
    quarantined_at TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.pipeline_quarantine
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT
    run_id,
    status,
    rows_inserted,
    rows_updated,
    rows_quarantined,
    error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND batch_id = 'full-sample-20261010-01'
  AND input_parameter LIKE '%table=silver_movies_cleansed%'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

quarantine_table = "workspace.tmdb_lakehouse.pipeline_quarantine"
test_batch_id = f"quarantine-test-{uuid4()}"

# Use one parsed Bronze record, but make its title invalid and mark it as test-only.
synthetic_invalid_df = (
    parsed_movies_df
    .limit(1)
    .withColumn("batch_id", F.lit(test_batch_id))
    .withColumn("title", F.lit(None).cast("string"))
    .withColumn(
        "validation_reason",
        F.lit("TEST ONLY: synthetic movie title is missing"),
    )
)

if synthetic_invalid_df.count() != 1:
    raise ValueError("Could not construct exactly one synthetic quarantine test record")

quarantine_test_source_df = (
    synthetic_invalid_df
    .withColumn("record_key", F.col("bronze_movie_id").cast("string"))
    .withColumn(
        "record_hash",
        F.sha2(
            F.to_json(F.struct("bronze_movie_id", "id", "title")),
            256,
        ),
    )
    .withColumn(
        "quarantine_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.lit("Bronze-to-Silver"),
                F.col("batch_id"),
                F.coalesce(F.col("source_file"), F.lit("")),
                F.col("record_key"),
                F.col("record_hash"),
            ),
            256,
        ),
    )
    .select(
        "quarantine_id",
        F.lit("Bronze-to-Silver").alias("layer"),
        "batch_id",
        "source_file",
        "source_extracted_at",
        "record_key",
        F.col("validation_reason").alias("reason"),
        "record_hash",
        F.current_timestamp().alias("quarantined_at"),
    )
)

quarantine_test_source_df.createOrReplaceTempView("quarantine_test_source")

merge_test_quarantine_sql = f"""
MERGE INTO {quarantine_table} AS target
USING quarantine_test_source AS source
ON target.quarantine_id = source.quarantine_id

WHEN NOT MATCHED THEN INSERT (
    quarantine_id,
    layer,
    batch_id,
    source_file,
    source_extracted_at,
    record_key,
    reason,
    record_hash,
    quarantined_at
) VALUES (
    source.quarantine_id,
    source.layer,
    source.batch_id,
    source.source_file,
    source.source_extracted_at,
    source.record_key,
    source.reason,
    source.record_hash,
    source.quarantined_at
)
"""

# Run twice to confirm duplicate-safe quarantine behavior.
spark.sql(merge_test_quarantine_sql)
spark.sql(merge_test_quarantine_sql)

test_rows = (
    spark.table(quarantine_table)
    .filter(F.col("batch_id") == test_batch_id)
    .count()
)

print("Synthetic invalid record quarantined exactly once:", test_rows == 1)
print("Synthetic quarantine test rows before cleanup:", test_rows)

if test_rows != 1:
    raise AssertionError(
        f"Expected exactly one test quarantine row; found {test_rows}"
    )

# Remove only the row created by this test batch.
spark.sql(f"""
DELETE FROM {quarantine_table}
WHERE batch_id = '{test_batch_id}'
""")

remaining_test_rows = (
    spark.table(quarantine_table)
    .filter(F.col("batch_id") == test_batch_id)
    .count()
)

print("Synthetic quarantine test rows after cleanup:", remaining_test_rows)

if remaining_test_rows != 0:
    raise AssertionError("Synthetic quarantine test cleanup did not complete")

In [0]:
from pyspark.sql import functions as F

drift_test_df = spark.createDataFrame([
    (
        "extra-field",
        '{"id":987654321,"title":"Schema drift test","new_source_field":"test"}',
    ),
    (
        "wrong-type",
        '{"id":"not-a-number","title":"Schema drift test"}',
    ),
], ["test_case", "raw_payload"])

drift_results_df = (
    drift_test_df
    .withColumn(
        "source_keys",
        F.expr("json_object_keys(raw_payload)"),
    )
    .withColumn(
        "parsed_movie",
        F.from_json("raw_payload", movie_schema),
    )
    .select(
        "test_case",
        "source_keys",
        F.col("parsed_movie.id").alias("parsed_id"),
        F.col("parsed_movie.title").alias("parsed_title"),
    )
)

drift_results_df.show(truncate=False)

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

quarantine_table = "workspace.tmdb_lakehouse.pipeline_quarantine"
test_batch_id = f"schema-drift-test-{uuid4()}"

drift_test_input_df = spark.createDataFrame(
    [
        (
            "additive-field",
            987654321,
            '{"id":987654321,"title":"Schema drift test","new_source_field":"retained in raw payload"}',
        ),
        (
            "incompatible-type",
            987654322,
            '{"id":"not-a-number","title":"Schema drift test"}',
        ),
    ],
    ["test_case", "bronze_movie_id", "raw_payload"],
)

drift_parsed_for_quarantine_df = (
    drift_test_input_df
    .withColumn("movie", F.from_json("raw_payload", movie_schema))
    .select(
        "test_case",
        "bronze_movie_id",
        F.col("movie.id").alias("id"),
        F.col("movie.title").alias("title"),
    )
    .withColumn("batch_id", F.lit(test_batch_id))
    .withColumn("source_file", F.lit("synthetic-schema-drift-test"))
    .withColumn("source_extracted_at", F.current_timestamp())
    .withColumn(
        "validation_reason",
        F.when(F.col("id").isNull(), F.lit("Parsed movie ID is missing"))
        .when(
            F.col("id") != F.col("bronze_movie_id"),
            F.lit("Parsed movie ID does not match Bronze movie ID"),
        )
        .when(
            F.col("title").isNull() | (F.length(F.trim(F.col("title"))) == 0),
            F.lit("Movie title is missing or blank"),
        ),
    )
)

invalid_drift_df = drift_parsed_for_quarantine_df.filter(
    F.col("validation_reason").isNotNull()
)
valid_drift_count = drift_parsed_for_quarantine_df.filter(
    F.col("validation_reason").isNull()
).count()

quarantine_drift_source_df = (
    invalid_drift_df
    .withColumn("record_key", F.col("bronze_movie_id").cast("string"))
    .withColumn(
        "record_hash",
        F.sha2(
            F.to_json(F.struct("bronze_movie_id", "id", "title")),
            256,
        ),
    )
    .withColumn(
        "quarantine_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.lit("Bronze-to-Silver"),
                F.col("batch_id"),
                F.col("source_file"),
                F.col("record_key"),
                F.col("record_hash"),
            ),
            256,
        ),
    )
    .select(
        "quarantine_id",
        F.lit("Bronze-to-Silver").alias("layer"),
        "batch_id",
        "source_file",
        "source_extracted_at",
        "record_key",
        F.col("validation_reason").alias("reason"),
        "record_hash",
        F.current_timestamp().alias("quarantined_at"),
    )
)

quarantine_drift_source_df.createOrReplaceTempView(
    "schema_drift_quarantine_source"
)

merge_drift_quarantine_sql = f"""
MERGE INTO {quarantine_table} AS target
USING schema_drift_quarantine_source AS source
ON target.quarantine_id = source.quarantine_id
WHEN NOT MATCHED THEN INSERT (
    quarantine_id,
    layer,
    batch_id,
    source_file,
    source_extracted_at,
    record_key,
    reason,
    record_hash,
    quarantined_at
) VALUES (
    source.quarantine_id,
    source.layer,
    source.batch_id,
    source.source_file,
    source.source_extracted_at,
    source.record_key,
    source.reason,
    source.record_hash,
    source.quarantined_at
)
"""

try:
    spark.sql(merge_drift_quarantine_sql)
    spark.sql(merge_drift_quarantine_sql)

    quarantined_drift_count = (
        spark.table(quarantine_table)
        .filter(F.col("batch_id") == test_batch_id)
        .count()
    )

    print("Synthetic additive-field records accepted:", valid_drift_count)
    print("Synthetic incompatible-type records quarantined:", quarantined_drift_count)
    print("Expected exactly one incompatible record:", quarantined_drift_count == 1)

    if valid_drift_count != 1 or quarantined_drift_count != 1:
        raise AssertionError("Schema-drift handling did not produce the expected counts")

finally:
    spark.sql(f"""
        DELETE FROM {quarantine_table}
        WHERE batch_id = '{test_batch_id}'
    """)
    print(
        "Synthetic drift test rows remaining:",
        spark.table(quarantine_table)
        .filter(F.col("batch_id") == test_batch_id)
        .count(),
    )

In [0]:
from pyspark.sql import functions as F

bronze_table = "workspace.tmdb_lakehouse.bronze_movie_records"

full_batch_id = "full-sample-20261010-01"
incremental_batch_id = "incremental-sample-20261010-01"

full_df = spark.table(bronze_table).filter(
    F.col("batch_id") == full_batch_id
)
incremental_df = spark.table(bronze_table).filter(
    F.col("batch_id") == incremental_batch_id
)

print("Full batch rows:", full_df.count())
print("Incremental batch rows:", incremental_df.count())

print("Full batch snapshot metadata:")
full_df.select(
    "load_type",
    "source_extracted_at",
    "window_start",
    "window_end",
).distinct().show(truncate=False)

print("Incremental batch snapshot metadata:")
incremental_df.select(
    "load_type",
    "source_extracted_at",
    "window_start",
    "window_end",
).distinct().show(truncate=False)

overlap_count = (
    full_df.select("movie_id").distinct()
    .join(
        incremental_df.select("movie_id").distinct(),
        on="movie_id",
        how="inner",
    )
    .count()
)
print("Movie IDs present in both batches:", overlap_count)

In [0]:
spark.sql("""
UPDATE workspace.tmdb_lakehouse.bronze_movie_records
SET source_extracted_at = TIMESTAMP '2026-10-10 00:01:00'
WHERE batch_id = 'incremental-sample-20261010-01'
""")

spark.table("workspace.tmdb_lakehouse.bronze_movie_records") \
    .filter("batch_id IN ('full-sample-20261010-01', 'incremental-sample-20261010-01')") \
    .select("batch_id", "load_type", "source_extracted_at") \
    .distinct() \
    .orderBy("source_extracted_at") \
    .show(truncate=False)

In [0]:
%sql
WITH ranked_runs AS (
    SELECT
        run_id,
        layer,
        load_type,
        batch_id,
        input_parameter,
        start_time,
        status,
        rows_inserted,
        rows_updated,
        rows_quarantined,
        error_details,
        ROW_NUMBER() OVER (
            PARTITION BY layer, input_parameter
            ORDER BY start_time DESC
        ) AS run_rank
    FROM workspace.tmdb_lakehouse.pipeline_execution_logs
    WHERE batch_id = 'incremental-sample-20261010-01'
)
SELECT
    layer,
    load_type,
    batch_id,
    input_parameter,
    start_time,
    status,
    rows_inserted,
    rows_updated,
    rows_quarantined,
    error_details
FROM ranked_runs
WHERE run_rank = 1
ORDER BY layer, input_parameter;